<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/0812_ETRIDATA_segformer_%EC%B0%A8%EC%84%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

세그멘테이션으로 차선 인식 데이터를 etri_ai에서 가지고 옴

In [ ]:
# 1단계: 환경 설정 및 데이터 전처리 (전체 데이터 사용)
# JSON 라벨 파일을 마스크 이미지로 변환

# 필요한 라이브러리 설치
!pip install -q opencv-python numpy

# 기본 라이브러리 임포트
import os
import json
import numpy as np
import cv2

print("라이브러리 로드 완료!")

# 압축 파일 해제
print("압축 파일 해제 중...")

# MonoCameraSemanticSegmentation.zip 해제
if os.path.exists('/content/sample_data/MonoCameraSemanticSegmentation.zip'):
    !cd /content/sample_data && unzip -q MonoCameraSemanticSegmentation.zip
    print("✅ MonoCameraSemanticSegmentation.zip 해제 완료!")

# labels.zip 해제
if os.path.exists('/content/sample_data/labels.zip'):
    !cd /content/sample_data && unzip -q labels.zip
    print("✅ labels.zip 해제 완료!")

# 🔍 실제 폴더 구조 확인
print("\n🔍 실제 폴더 구조 확인:")
print("sample_data 폴더 내용:")
!ls -la /content/sample_data/

# 🎯 실제 이미지 폴더 찾기
possible_image_paths = [
    '/content/sample_data/JPEGImages_mosaic',  # 바로 sample_data 아래
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages',
]

IMAGE_DIR = None
for path in possible_image_paths:
    if os.path.exists(path):
        IMAGE_DIR = path
        print(f"✅ 이미지 폴더 찾음: {path}")
        # 몇 개 파일이 있는지 확인
        png_files = [f for f in os.listdir(path) if f.endswith('.png')]
        print(f"   📁 PNG 파일 수: {len(png_files)}개")
        break

if IMAGE_DIR is None:
    print("❌ 이미지 폴더를 찾을 수 없습니다!")
    print("🔍 sample_data 하위 폴더 전체 탐색:")
    for item in os.listdir('/content/sample_data/'):
        item_path = os.path.join('/content/sample_data/', item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
            try:
                files = os.listdir(item_path)
                png_count = len([f for f in files if f.endswith('.png')])
                if png_count > 0:
                    print(f"     🖼️ PNG 파일 {png_count}개 발견!")
                    IMAGE_DIR = item_path
            except:
                pass

# 경로 설정
LABEL_DIR = '/content/sample_data/labels'
MASK_DIR = '/content/sample_data/masks_final'

class_map = {
    "background": 0,
    "lane": 1,
}

print("경로 설정 완료!")
print(f"이미지 폴더: {IMAGE_DIR}")
print(f"라벨 폴더: {LABEL_DIR}")
print(f"마스크 저장 폴더: {MASK_DIR}")

# 전체 데이터 현황 파악
print("\n📊 전체 데이터 현황 파악 중...")

if os.path.exists(IMAGE_DIR):
    all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]
    daeduk_images = [f for f in all_images if f.startswith('Daeduk')]
    sangam_images = [f for f in all_images if f.startswith('SangamDMC')]

    print(f"🖼️ 이미지 파일 현황:")
    print(f"  - Daeduk: {len(daeduk_images)}개")
    print(f"  - SangamDMC: {len(sangam_images)}개")
    print(f"  - 총 이미지: {len(all_images)}개")

if os.path.exists(LABEL_DIR):
    all_labels = [f for f in os.listdir(LABEL_DIR) if f.endswith('.json')]
    daeduk_labels = [f for f in all_labels if f.startswith('Daeduk')]
    sangam_labels = [f for f in all_labels if f.startswith('SangamDMC')]

    print(f"🏷️ 라벨 파일 현황:")
    print(f"  - Daeduk: {len(daeduk_labels)}개")
    print(f"  - SangamDMC: {len(sangam_labels)}개")
    print(f"  - 총 라벨: {len(all_labels)}개")

# 매칭 가능한 데이터 분석
print(f"\n🎯 매칭 분석:")

# 변수 초기화
matched_basenames = []
daeduk_matched = []
sangam_matched = []

if IMAGE_DIR and os.path.exists(IMAGE_DIR) and os.path.exists(LABEL_DIR) and len(all_images) > 0 and len(all_labels) > 0:
    # 이미지 베이스명들
    image_basenames = [f.replace('_leftImg8bit.png', '') for f in all_images]
    # 라벨 베이스명들
    label_basenames = [f.replace('_gtFine_polygons.json', '') for f in all_labels]

    # 매칭되는 것들 찾기
    matched_basenames = list(set(image_basenames) & set(label_basenames))

    daeduk_matched = [name for name in matched_basenames if name.startswith('Daeduk')]
    sangam_matched = [name for name in matched_basenames if name.startswith('SangamDMC')]

    print(f"  - Daeduk 매칭: {len(daeduk_matched)}개")
    print(f"  - SangamDMC 매칭: {len(sangam_matched)}개")
    print(f"  - 총 사용가능: {len(matched_basenames)}개")
else:
    if not IMAGE_DIR or not os.path.exists(IMAGE_DIR):
        print("❌ 이미지 폴더를 찾을 수 없습니다!")
    elif not os.path.exists(LABEL_DIR):
        print("❌ 라벨 폴더를 찾을 수 없습니다!")
    elif len(all_images) == 0:
        print("❌ 이미지 파일이 없습니다!")
    elif len(all_labels) == 0:
        print("❌ 라벨 파일이 없습니다!")

print(f"🎯 처리할 총 데이터: {len(matched_basenames)}개")

# 마스크 이미지 생성
print("\n🚀 전체 데이터로 마스크 이미지 생성 시작...")
os.makedirs(MASK_DIR, exist_ok=True)

if len(matched_basenames) == 0:
    print("❌ 처리할 데이터가 없습니다! 경로를 확인해주세요.")
    print("\n✨ 1단계 완료 (데이터 없음)")
else:
    processed_count = 0
    error_count = 0

    # 매칭된 데이터만 처리
    for i, base_name in enumerate(matched_basenames):
        # 파일 경로 구성
        json_file = f"{base_name}_gtFine_polygons.json"
        image_file = f"{base_name}_leftImg8bit.png"

        label_path = os.path.join(LABEL_DIR, json_file)
        image_path = os.path.join(IMAGE_DIR, image_file)

        # 이미지 로드
        image = cv2.imread(image_path)
        if image is None:
            print(f"  ❌ {i+1}/{len(matched_basenames)} 에러: 이미지 로드 실패 ({image_file})")
            error_count += 1
            continue

        # 이미지 크기 가져오기
        height, width, _ = image.shape

        # JSON 라벨 파일 읽기
        try:
            with open(label_path, 'r') as f:
                data = json.load(f)
        except:
            print(f"  ❌ {i+1}/{len(matched_basenames)} 에러: JSON 파일 읽기 실패 ({json_file})")
            error_count += 1
            continue

        # 빈 마스크 생성
        mask = np.zeros((height, width), dtype=np.uint8)

        # 차선 폴리곤을 마스크에 그리기
        lane_count = 0
        for obj in data['objects']:
            label = obj['label']
            points = np.array(obj['polygon'], dtype=np.int32)

            # 🎯 실제 데이터셋에서 사용하는 차선 관련 라벨들
            lane_labels = [
                'whdot',           # 흰색 점선 (가장 많음)
                'yedot',           # 노란색 점선
                'blsol',           # 파란색 실선
                'yesol',           # 노란색 실선
                'general road mark' # 일반 도로 표시
            ]

            if label in lane_labels:
                cv2.fillPoly(mask, [points], color=class_map['lane'])
                lane_count += 1

        # 마스크 저장
        mask_file_name = image_file.replace('.png', '_mask.png')
        mask_save_path = os.path.join(MASK_DIR, mask_file_name)
        cv2.imwrite(mask_save_path, mask)

        processed_count += 1
        area_type = "🟦 Daeduk" if base_name.startswith('Daeduk') else "🟩 SangamDMC"
        print(f"  ✅ {i+1}/{len(matched_basenames)} {area_type}: {mask_file_name} (차선 {lane_count}개)")

    print(f"\n🎉 전체 데이터 마스크 생성 완료!")
    print(f"✅ 성공: {processed_count}개 (Daeduk + SangamDMC)")
    print(f"❌ 실패: {error_count}개")
    print(f"📁 저장 위치: {MASK_DIR}")

    # 생성된 파일 확인
    mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith('_mask.png')]
    print(f"📊 생성된 마스크 파일 수: {len(mask_files)}개")

    print("\n✨ 1단계 완료! 다음 단계로 진행하세요.")

2단계

In [ ]:
# 2단계: 데이터셋 생성 및 분할 (경로 수정된 버전)
# 이미지와 마스크를 Hugging Face 데이터셋으로 변환

# 딥러닝 라이브러리 설치
!pip install -q transformers datasets evaluate accelerate

# 필요한 라이브러리 임포트
import os
from datasets import Dataset, DatasetDict
from datasets import Image as HFImage
from PIL import Image

print("딥러닝 라이브러리 로드 완료!")

# 🔍 실제 경로 자동 탐지
print("\n🔍 실제 폴더 위치 확인 중...")

# 가능한 이미지 폴더 경로들
possible_image_paths = [
    '/content/sample_data/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages',
]

IMAGE_DIR = None
for path in possible_image_paths:
    if os.path.exists(path):
        IMAGE_DIR = path
        png_files = [f for f in os.listdir(path) if f.endswith('.png')]
        print(f"✅ 이미지 폴더 찾음: {path}")
        print(f"   📁 PNG 파일 수: {len(png_files)}개")
        break

if IMAGE_DIR is None:
    print("❌ 기본 경로에서 이미지 폴더를 찾을 수 없습니다!")
    print("🔍 sample_data 하위 폴더 전체 탐색:")

    for item in os.listdir('/content/sample_data/'):
        item_path = os.path.join('/content/sample_data/', item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
            try:
                files = os.listdir(item_path)
                png_count = len([f for f in files if f.endswith('.png')])
                if png_count > 0:
                    print(f"     🖼️ PNG 파일 {png_count}개 발견!")
                    IMAGE_DIR = item_path
                    break
            except:
                pass

# 마스크 폴더는 고정
MASK_DIR = '/content/sample_data/masks_final'

print(f"\n📁 최종 경로 설정:")
print(f"  - 이미지: {IMAGE_DIR}")
print(f"  - 마스크: {MASK_DIR}")

# 경로 유효성 확인
if IMAGE_DIR is None or not os.path.exists(IMAGE_DIR):
    print("\n❌ 이미지 폴더를 찾을 수 없습니다!")
    print("💡 해결 방법:")
    print("1. 1단계(압축 해제 및 마스크 생성)를 먼저 실행하세요")
    print("2. 또는 다음 명령어로 수동 압축 해제:")
    print("   !cd /content/sample_data && unzip -q MonoCameraSemanticSegmentation.zip")
    exit()

if not os.path.exists(MASK_DIR):
    print(f"\n❌ 마스크 폴더가 없습니다: {MASK_DIR}")
    print("💡 1단계(마스크 생성)를 먼저 실행하세요!")
    exit()

# 클래스 정의
id2label = {0: "background", 1: "lane"}
label2id = {v: k for k, v in id2label.items()}

print("클래스 설정:")
print(f"  - {id2label}")

# 이미지와 마스크 파일 경로 수집 (전체 데이터 사용)
print("\n파일 경로 수집 중...")

# 모든 이미지와 마스크 파일 수집 (Daeduk + SangamDMC)
if os.path.exists(IMAGE_DIR):
    image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]
else:
    image_files = []

if os.path.exists(MASK_DIR):
    mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith('_mask.png')]
else:
    mask_files = []

# 전체 경로로 변환
image_paths = sorted([os.path.join(IMAGE_DIR, f) for f in image_files])
mask_paths = sorted([os.path.join(MASK_DIR, f) for f in mask_files])

# 데이터 현황 분석
daeduk_images = [p for p in image_paths if 'Daeduk' in os.path.basename(p)]
sangam_images = [p for p in image_paths if 'SangamDMC' in os.path.basename(p)]
daeduk_masks = [p for p in mask_paths if 'Daeduk' in os.path.basename(p)]
sangam_masks = [p for p in mask_paths if 'SangamDMC' in os.path.basename(p)]

print(f"📊 수집된 파일 (전체 데이터):")
print(f"  🖼️ 이미지: {len(image_paths)}개")
print(f"    - 🟦 Daeduk: {len(daeduk_images)}개")
print(f"    - 🟩 SangamDMC: {len(sangam_images)}개")
print(f"  🎭 마스크: {len(mask_paths)}개")
print(f"    - 🟦 Daeduk: {len(daeduk_masks)}개")
print(f"    - 🟩 SangamDMC: {len(sangam_masks)}개")

# 파일 수가 일치하는지 확인
if len(image_paths) != len(mask_paths):
    print("⚠️  경고: 이미지와 마스크 파일 수가 일치하지 않습니다!")
    print(f"이미지: {len(image_paths)}개, 마스크: {len(mask_paths)}개")

# 데이터가 없으면 중단
if len(image_paths) == 0 or len(mask_paths) == 0:
    print("\n❌ 데이터가 없습니다!")
    print(f"이미지: {len(image_paths)}개, 마스크: {len(mask_paths)}개")
    print("💡 1단계를 먼저 실행해서 데이터를 준비하세요!")
    exit()

# 데이터셋 생성
print("\n데이터셋 생성 중...")
dataset = Dataset.from_dict({
    "image": image_paths,
    "label": mask_paths,
})

# 이미지 타입으로 변환 (Hugging Face 형식)
dataset = dataset.cast_column("image", HFImage())
dataset = dataset.cast_column("label", HFImage())

print("✅ 기본 데이터셋 생성 완료!")

# 훈련/테스트 분할 (8:2 비율)
print("\n데이터셋 분할 중...")
train_test_split = dataset.train_test_split(test_size=0.2, seed=42)

# 최종 데이터셋 구성
ds = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test'],
})

print("\n🎉 데이터셋 분할 완료!")
print(f"📊 최종 데이터셋 구성:")
print(f"  - 훈련용: {len(ds['train'])}개")
print(f"  - 테스트용: {len(ds['test'])}개")
print(f"  - 총합: {len(dataset)}개")

# 데이터셋 구조 확인
print(f"\n📋 데이터셋 구조:")
print(ds)

# 샘플 데이터 확인
if len(ds['train']) > 0:
    print(f"\n🔍 샘플 데이터 확인:")
    sample = ds['train'][0]
    print(f"  - 이미지 크기: {sample['image'].size}")
    print(f"  - 라벨 크기: {sample['label'].size}")
else:
    print(f"\n❌ 훈련 데이터가 없습니다!")

print("\n✨ 2단계 완료! 다음 단계로 진행하세요.")

# 다음 단계에서 사용할 변수들 저장
print("\n💾 다음 단계를 위한 변수들:")
print("ds = 생성된 데이터셋")
print("id2label, label2id = 클래스 매핑")
print(f"실제 데이터 수: 훈련 {len(ds['train'])}개, 테스트 {len(ds['test'])}개")

3단

In [ ]:
# 3단계: 모델 및 전처리 설정
# SegFormer 모델과 이미지 전처리 파이프라인 구성

# 필요한 라이브러리 임포트
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from torchvision.transforms import ColorJitter
import torch

print("모델 라이브러리 로드 완료!")

# 클래스 설정 (2단계에서 가져옴)
id2label = {0: "background", 1: "lane"}
label2id = {v: k for k, v in id2label.items()}

print("클래스 매핑:")
print(f"  - ID to Label: {id2label}")
print(f"  - Label to ID: {label2id}")

# SegFormer 이미지 전처리기 설정
print("\n이미지 전처리기 설정 중...")
try:
    image_processor = SegformerImageProcessor.from_pretrained(
        "nvidia/segformer-b0-finetuned-ade-512-512",
        id2label=id2label,
        label2id=label2id,
        do_reduce_labels=False  # 라벨 자동 변환 비활성화
    )
    print("✅ 이미지 전처리기 설정 완료!")
except Exception as e:
    print(f"❌ 전처리기 설정 오류: {e}")
    print("인터넷 연결을 확인하고 다시 시도하세요.")

# 데이터 증강 설정 (훈련 시에만 사용)
print("\n데이터 증강 설정 중...")
jitter = ColorJitter(
    brightness=0.25,  # 밝기 변화
    contrast=0.25,    # 대비 변화
    saturation=0.25,  # 채도 변화
    hue=0.1          # 색조 변화
)

def train_transforms(example_batch):
    """훈련용 데이터 변환 (데이터 증강 포함)"""
    # 이미지에 데이터 증강 적용
    images = [jitter(x.convert("RGB")) for x in example_batch['image']]
    # 라벨은 그대로 유지
    labels = [x for x in example_batch['label']]

    # 전처리기로 배치 처리
    inputs = image_processor(images, labels, return_tensors="pt")
    return inputs

def val_transforms(example_batch):
    """검증용 데이터 변환 (데이터 증강 없음)"""
    # 이미지를 RGB로만 변환
    images = [x.convert("RGB") for x in example_batch['image']]
    # 라벨은 그대로 유지
    labels = [x for x in example_batch['label']]

    # 전처리기로 배치 처리
    inputs = image_processor(images, labels, return_tensors="pt")
    return inputs

print("✅ 데이터 변환 함수 정의 완료!")

# SegFormer 모델 로드
print("\n🤖 SegFormer 모델 로드 중...")
try:
    model = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/mit-b0",  # Mix Transformer B0 backbone 사용
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True  # 클래스 수가 다를 때 크기 무시
    )
    print("✅ SegFormer 모델 로드 완료!")

    # 모델 정보 출력
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"📊 모델 정보:")
    print(f"  - Backbone: Mix Transformer B0")
    print(f"  - 클래스 수: {len(id2label)}개")
    print(f"  - 총 파라미터: {total_params/1e6:.1f}M개")
    print(f"  - 훈련 가능 파라미터: {trainable_params/1e6:.1f}M개")

except Exception as e:
    print(f"❌ 모델 로드 오류: {e}")
    print("인터넷 연결을 확인하고 다시 시도하세요.")

# GPU 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🔧 사용 장치: {device}")

if torch.cuda.is_available():
    print(f"  - GPU: {torch.cuda.get_device_name()}")
    print(f"  - CUDA 버전: {torch.version.cuda}")

    # GPU 메모리 정보
    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    gpu_memory_gb = gpu_memory / 1e9
    print(f"  - GPU 메모리: {gpu_memory_gb:.1f} GB")

    # 권장 배치 크기
    if gpu_memory_gb >= 16:
        recommended_batch = 4
    elif gpu_memory_gb >= 8:
        recommended_batch = 2
    else:
        recommended_batch = 1
    print(f"  - 권장 배치 크기: {recommended_batch}")

else:
    print("  - CPU 모드로 실행됩니다 (훈련이 매우 느릴 수 있습니다)")

# 모델을 GPU로 이동 (가능한 경우)
if torch.cuda.is_available():
    try:
        model = model.to(device)
        print("✅ 모델을 GPU로 이동 완료!")
    except Exception as e:
        print(f"⚠️ GPU 이동 실패, CPU 사용: {e}")

print("\n✨ 3단계 완료! 다음 단계로 진행하세요.")

# 다음 단계에서 사용할 변수들
print("\n💾 다음 단계를 위한 주요 변수들:")
print("model = SegFormer 모델")
print("image_processor = 이미지 전처리기")
print("train_transforms, val_transforms = 데이터 변환 함수")
print("id2label, label2id = 클래스 매핑")
print(f"device = {device}")

# 메모리 최적화 팁
print(f"\n💡 메모리 최적화 팁:")
print(f"  - GPU 메모리 부족 시: 배치 크기를 1로 줄이세요")
print(f"  - 훈련 속도 향상: eval_steps를 늘리세요 (예: 500)")
print(f"  - 더 정확한 평가: eval_steps를 줄이세요 (예: 100)")

# 다음 단계 안내
print(f"\n🎯 다음 단계 안내:")
print(f"  4단계: 훈련 설정 및 메트릭")
print(f"  5단계: 실제 모델 훈련")

In [ ]:
# 4단계: 훈련 설정 및 메트릭
# 평가 메트릭과 훈련 파라미터 설정

# 필요한 라이브러리 설치 및 임포트
!pip install -q evaluate torch torchvision

import torch
from torch import nn
import evaluate
from transformers import TrainingArguments, Trainer

print("훈련 라이브러리 로드 완료!")

# 평가 메트릭 설정
print("\n📊 평가 메트릭 설정 중...")
try:
    metric = evaluate.load("mean_iou")
    print("✅ mIoU 메트릭 로드 완료!")
except Exception as e:
    print(f"❌ 메트릭 로드 오류: {e}")
    print("인터넷 연결을 확인하고 다시 시도하세요.")

def compute_metrics(eval_pred):
    """모델 성능 평가 함수"""
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)

        # 예측 결과를 원본 이미지 크기로 리사이즈
        logits_resized = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],  # (height, width)
            mode="bilinear",
            align_corners=False,
        )

        # 가장 높은 확률의 클래스를 예측값으로 선택
        pred_labels = logits_resized.argmax(dim=1).numpy()

        # mIoU 및 정확도 계산
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=len(id2label),
            ignore_index=255,  # 무시할 픽셀 인덱스
            reduce_labels=False,
        )

        return {
            "mean_iou": metrics["mean_iou"],
            "mean_accuracy": metrics["mean_accuracy"],
        }

print("✅ 평가 메트릭 설정 완료!")

# 훈련 파라미터 설정
print("\n⚙️ 훈련 파라미터 설정 중...")

# GPU 메모리에 따른 배치 크기 자동 조정
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_memory >= 16:
        batch_size = 4
        print(f"🔧 고용량 GPU 감지 ({gpu_memory:.1f}GB): 배치 크기 {batch_size}")
    elif gpu_memory >= 8:
        batch_size = 2
        print(f"🔧 중용량 GPU 감지 ({gpu_memory:.1f}GB): 배치 크기 {batch_size}")
    else:
        batch_size = 1
        print(f"🔧 저용량 GPU 감지 ({gpu_memory:.1f}GB): 배치 크기 {batch_size}")
else:
    batch_size = 1
    print("🔧 CPU 모드: 배치 크기 1")

training_args = TrainingArguments(
    # 기본 설정
    output_dir="./segformer-lane-detection",

    # 학습률 및 에포크
    learning_rate=6e-5,
    num_train_epochs=20,

    # 배치 크기 (GPU 메모리에 따라 자동 조정)
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,

    # 저장 설정
    save_total_limit=3,  # 최대 3개 체크포인트만 유지
    save_strategy="steps",
    save_steps=200,

    # 평가 설정 (수정된 파라미터명)
    eval_strategy="steps",  # evaluation_strategy → eval_strategy
    eval_steps=200,

    # 로깅
    logging_dir="./logs",
    logging_steps=100,

    # 기타 설정
    remove_unused_columns=False,  # 사용하지 않는 컬럼 제거 안함
    push_to_hub=False,  # Hugging Face Hub에 업로드 안함

    # 성능 최적화
    dataloader_pin_memory=True,
    dataloader_num_workers=2,

    # 조기 종료 (옵션)
    load_best_model_at_end=True,
    metric_for_best_model="mean_iou",
    greater_is_better=True,

    # 추가 설정
    report_to=None,  # wandb 등 로깅 비활성화
    seed=42,  # 재현 가능한 결과를 위한 시드
)

print("✅ 훈련 파라미터 설정 완료!")

# 설정 요약 출력
print(f"\n📋 주요 훈련 설정:")
print(f"  - 학습률: {training_args.learning_rate}")
print(f"  - 에포크: {training_args.num_train_epochs}")
print(f"  - 배치 크기: {training_args.per_device_train_batch_size}")
print(f"  - 평가 주기: {training_args.eval_steps} 스텝마다")
print(f"  - 저장 위치: {training_args.output_dir}")
print(f"  - 로그 위치: {training_args.logging_dir}")

# 예상 훈련 시간 계산 (대략적)
if 'ds' in globals():
    total_steps = (len(ds['train']) // training_args.per_device_train_batch_size) * training_args.num_train_epochs
    estimated_time = total_steps * 2 / 60  # 대략 스텝당 2초 가정
    print(f"  - 예상 총 스텝: {total_steps}")
    print(f"  - 예상 훈련 시간: 약 {estimated_time:.0f}분")
else:
    print(f"  - ds 변수가 없어서 시간 계산 불가")

# 메모리 최적화 및 성능 팁
print(f"\n💡 메모리 최적화 팁:")
print(f"  - GPU 메모리 부족 시: 배치 크기를 1로 줄이세요")
print(f"  - 더 빠른 훈련: eval_steps를 500으로 늘리세요")
print(f"  - 더 정확한 평가: eval_steps를 100으로 줄이세요")
print(f"  - 디스크 공간 절약: save_total_limit를 1로 줄이세요")

# GPU 메모리 정보 (사용 가능한 경우)
if torch.cuda.is_available():
    print(f"\n🔧 현재 GPU 상태:")
    print(f"  - 총 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if hasattr(torch.cuda, 'mem_get_info'):
        free_memory, total_memory = torch.cuda.mem_get_info()
        print(f"  - 사용 가능: {free_memory / 1e9:.1f} GB")
        print(f"  - 사용 중: {(total_memory - free_memory) / 1e9:.1f} GB")

print("\n✨ 4단계 완료! 다음 단계로 진행하세요.")

# 다음 단계에서 사용할 변수들
print("\n💾 다음 단계를 위한 주요 변수들:")
print("training_args = 훈련 파라미터")
print("compute_metrics = 평가 함수")
print("metric = mIoU 메트릭")

# 다음 단계 안내
print(f"\n🎯 다음 단계 안내:")
print(f"  5단계: 실제 모델 훈련 및 평가")
print(f"  - 데이터셋 전처리 적용")
print(f"  - Trainer 설정")
print(f"  - 훈련 실행")
print(f"  - 최종 평가 및 모델 저장")

# 훈련 전 체크리스트
print(f"\n✅ 훈련 전 체크리스트:")
required_vars = ['ds', 'model', 'image_processor', 'train_transforms', 'val_transforms', 'id2label']
missing_vars = [var for var in required_vars if var not in globals()]

if missing_vars:
    print(f"❌ 누락된 변수들: {missing_vars}")
    print(f"이전 단계들을 먼저 실행해주세요!")
else:
    print(f"✅ 모든 필수 변수가 준비되었습니다!")
    print(f"5단계를 실행할 수 있습니다!")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# wandb 충돌 해결
import os
if "WANDB_DISABLED" in os.environ:
    del os.environ["WANDB_DISABLED"]

# 수정된 training_args로 다시 설정
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./segformer-lane-detection",
    learning_rate=6e-5,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_total_limit=3,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=100,
    remove_unused_columns=False,
    push_to_hub=False,
    report_to="none",  # 🔧 이 줄이 핵심!
    seed=42,
)

print("✅ training_args 수정 완료!")

In [ ]:
# 5단계: 훈련 실행 (간단 버전)
# 모든 설정을 통합하여 실제 훈련 시작

print("🔍 사전 확인 중...")

# 필수 변수들이 모두 있는지 확인
required_vars = ['ds', 'model', 'training_args', 'compute_metrics', 'train_transforms', 'val_transforms']
missing_vars = []

for var_name in required_vars:
    if var_name not in globals():
        missing_vars.append(var_name)

if missing_vars:
    print(f"❌ 누락된 변수들: {missing_vars}")
    print("이전 단계들을 먼저 실행해주세요!")
    print("필요한 단계:")
    if 'ds' in missing_vars:
        print("  - 2단계: 데이터셋 생성")
    if 'model' in missing_vars:
        print("  - 3단계: 모델 설정")
    if 'training_args' in missing_vars:
        print("  - 4단계: 훈련 설정")
else:
    print("✅ 모든 필수 변수들이 준비되었습니다!")

# 데이터셋에 전처리 함수 적용
print("\n📦 데이터셋 전처리 적용 중...")
ds["train"].set_transform(train_transforms)
ds["test"].set_transform(val_transforms)
print("✅ 데이터셋 전처리 적용 완료!")

# Trainer 설정
print("\n🎯 Trainer 설정 중...")
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    compute_metrics=compute_metrics,
)

print("✅ Trainer 설정 완료!")

# 훈련 시작 전 최종 정보 출력
print(f"\n📊 훈련 시작 전 최종 정보:")
print(f"  - 모델: SegFormer (MiT-B0)")
print(f"  - 훈련 데이터: {len(ds['train'])}개")
print(f"  - 테스트 데이터: {len(ds['test'])}개")
print(f"  - 클래스 수: {len(id2label)}개 (배경, 차선)")
print(f"  - 에포크: {training_args.num_train_epochs}")
print(f"  - 배치 크기: {training_args.per_device_train_batch_size}")

# 예상 훈련 시간 계산
total_steps = (len(ds['train']) // training_args.per_device_train_batch_size) * training_args.num_train_epochs
estimated_time = total_steps * 2 / 60  # 대략 스텝당 2초 가정
print(f"  - 예상 총 스텝: {total_steps}")
print(f"  - 예상 훈련 시간: 약 {estimated_time:.0f}분")

# 🚀 훈련 시작
print(f"\n🚀 훈련 시작!")
print("=" * 50)

try:
    # 실제 훈련 실행
    trainer.train()

    print("\n" + "=" * 50)
    print("🎉 훈련 완료!")

    # 최종 평가
    print(f"\n📊 최종 평가 중...")
    eval_results = trainer.evaluate()

    print(f"✅ 최종 평가 결과:")
    for key, value in eval_results.items():
        if 'eval_' in key:
            metric_name = key.replace('eval_', '')
            print(f"  - {metric_name}: {value:.4f}")

    # 모델 저장
    print(f"\n💾 모델 저장 중...")
    trainer.save_model()
    print(f"✅ 모델 저장 완료: {training_args.output_dir}")

    # 성능 해석
    final_iou = eval_results.get('eval_mean_iou', 0)
    final_acc = eval_results.get('eval_mean_accuracy', 0)

    print(f"\n🎯 성능 해석:")
    if final_iou > 0.7:
        print(f"  🟢 훌륭한 성능! (mIoU: {final_iou:.3f})")
    elif final_iou > 0.5:
        print(f"  🟡 괜찮은 성능! (mIoU: {final_iou:.3f})")
    else:
        print(f"  🔴 추가 훈련 필요 (mIoU: {final_iou:.3f})")

    print(f"\n📁 결과물:")
    print(f"  - 훈련된 모델: {training_args.output_dir}")
    print(f"  - 로그 파일: ./logs")

    # 성능 개선 팁
    print(f"\n💡 성능 개선 팁:")
    if final_iou < 0.6:
        print(f"  - 더 많은 에포크로 재훈련")
        print(f"  - 학습률 조정 (현재: {training_args.learning_rate})")
        print(f"  - 데이터 증강 강화")

except Exception as e:
    print(f"\n❌ 훈련 중 오류 발생:")
    print(f"Error: {str(e)}")
    print(f"\n💡 해결 방법:")
    print(f"  1. GPU 메모리 부족 → 배치 크기를 1로 줄이기")
    print(f"  2. CUDA 오류 → 런타임 재시작")
    print(f"  3. 데이터 오류 → 이전 단계들 다시 확인")

    # 메모리 부족 시 자동 해결 시도
    if "out of memory" in str(e).lower():
        print(f"\n🔧 자동 해결 시도: 배치 크기를 1로 줄여서 재시도")
        training_args.per_device_train_batch_size = 1
        training_args.per_device_eval_batch_size = 1

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=ds["train"],
            eval_dataset=ds["test"],
            compute_metrics=compute_metrics,
        )

        try:
            trainer.train()
            print("✅ 배치 크기 1로 훈련 성공!")
        except:
            print("❌ 배치 크기 1로도 실패. 런타임을 재시작하세요.")

print(f"\n✨ 5단계 완료! 차선 검출 모델 훈련이 끝났습니다!")

# 다음 단계 안내
print(f"\n🎯 훈련 완료 후 할 수 있는 것들:")
print(f"  1. 새로운 이미지로 테스트")
print(f"  2. 모델 성능 분석")
print(f"  3. 하이퍼파라미터 튜닝으로 성능 개선")
print(f"  4. 더 많은 데이터로 재훈련")

In [ ]:
trainer.train()  # ← 여기서 진짜 20번 훈련!